In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import os

In [10]:
DATA_PATH = "../data/raw/driver_risk.csv"

df = pd.read_csv(DATA_PATH, sep="\t")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Shape: (123, 14)


,Weather,Road_Type,Time_of_Day,Traffic_Density,Speed_Limit,Number_of_Vehicles,Driver_Alcohol,Accident_Severity,Road_Condition,Vehicle_Type,Driver_Age,Driver_Experience,Road_Light_Condition,Accident
0,Rainy,City Road,Morning,1.0,100.0,5.0,0.0,NaN,Wet,Car,51.0,48.0,Artificial Light,0.0
1,Clear,Rural Road,Night,NaN,120.0,3.0,0.0,Moderate,Wet,Truck,49.0,43.0,Artificial Light,0.0
2,Rainy,Highway,Evening,1.0,60.0,4.0,0.0,Low,Icy,Car,54.0,52.0,Artificial Light,0.0
3,Clear,City Road,Afternoon,2.0,60.0,3.0,0.0,Low,Under Construction,Bus,34.0,31.0,Daylight,0.0
4,Rainy,Highway,Morning,1.0,195.0,11.0,0.0,Low,Dry,Car,62.0,55.0,Artificial Light,1.0


In [11]:
DATA_PATH = "../data/raw/driver_risk.csv"

df = pd.read_csv(DATA_PATH, sep="\t")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Shape: (123, 14)


,Weather,Road_Type,Time_of_Day,Traffic_Density,Speed_Limit,Number_of_Vehicles,Driver_Alcohol,Accident_Severity,Road_Condition,Vehicle_Type,Driver_Age,Driver_Experience,Road_Light_Condition,Accident
0,Rainy,City Road,Morning,1.0,100.0,5.0,0.0,NaN,Wet,Car,51.0,48.0,Artificial Light,0.0
1,Clear,Rural Road,Night,NaN,120.0,3.0,0.0,Moderate,Wet,Truck,49.0,43.0,Artificial Light,0.0
2,Rainy,Highway,Evening,1.0,60.0,4.0,0.0,Low,Icy,Car,54.0,52.0,Artificial Light,0.0
3,Clear,City Road,Afternoon,2.0,60.0,3.0,0.0,Low,Under Construction,Bus,34.0,31.0,Daylight,0.0
4,Rainy,Highway,Morning,1.0,195.0,11.0,0.0,Low,Dry,Car,62.0,55.0,Artificial Light,1.0


In [12]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

print(df.columns.tolist())

['Weather', 'Road_Type', 'Time_of_Day', 'Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles', 'Driver_Alcohol', 'Accident_Severity', 'Road_Condition', 'Vehicle_Type', 'Driver_Age', 'Driver_Experience', 'Road_Light_Condition', 'Accident']


In [13]:
before = len(df)

df = df.dropna(how="all").reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after :", after)
print("Completely empty rows removed:", before - after)

Rows before: 123
Rows after : 123
Completely empty rows removed: 0


In [14]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [15]:
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (123, 14)


In [16]:
numerical_columns = [
    "Traffic_Density",
    "Speed_Limit",
    "Number_of_Vehicles",
    "Driver_Alcohol",
    "Driver_Age",
    "Driver_Experience",
    "Accident"
]

In [17]:
for col in numerical_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [18]:
missing_target = df["Accident"].isna().sum()

print("Missing target values:", missing_target)

Missing target values: 5


In [19]:
df = df.dropna(subset=["Accident"]).reset_index(drop=True)

print("Shape after removing missing target rows:", df.shape)

Shape after removing missing target rows: (118, 14)


In [20]:
print(df["Accident"].value_counts(dropna=False))

Accident
0.0    82
1.0    36
Name: count, dtype: int64


In [21]:
invalid_target = ~df["Accident"].isin([0, 1])

print("Invalid target rows:", invalid_target.sum())

Invalid target rows: 0


In [22]:
df["Accident"] = df["Accident"].astype(int)


In [23]:
X = df.drop(columns=["Accident"])
y = df["Accident"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (118, 13)
Target shape: (118,)


In [24]:
if "Accident_Severity" in X.columns:
    X = X.drop(columns=["Accident_Severity"])

print("Features after leakage check:")
print(X.columns.tolist())

Features after leakage check:
['Weather', 'Road_Type', 'Time_of_Day', 'Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles', 'Driver_Alcohol', 'Road_Condition', 'Vehicle_Type', 'Driver_Age', 'Driver_Experience', 'Road_Light_Condition']


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [26]:
print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training data: (94, 12)
Testing data : (24, 12)

Training target distribution:
Accident
0    0.691489
1    0.308511
Name: proportion, dtype: float64

Testing target distribution:
Accident
0    0.708333
1    0.291667
Name: proportion, dtype: float64


In [27]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles', 'Driver_Alcohol', 'Driver_Age', 'Driver_Experience']

Categorical features:
['Weather', 'Road_Type', 'Time_of_Day', 'Road_Condition', 'Vehicle_Type', 'Road_Light_Condition']


C:\Users\keert\AppData\Local\Temp\ipykernel_7416\1124402278.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [28]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [29]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

In [31]:
X_train_processed = preprocessor.fit_transform(X_train)

In [32]:
X_test_processed = preprocessor.transform(X_test)

In [35]:
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

Original training shape: (94, 12)
Processed training shape: (94, 30)
Original testing shape: (24, 12)
Processed testing shape: (24, 30)


In [36]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

for feature in feature_names:
    print(feature)

Number of processed features: 30
numerical__Traffic_Density
numerical__Speed_Limit
numerical__Number_of_Vehicles
numerical__Driver_Alcohol
numerical__Driver_Age
numerical__Driver_Experience
categorical__Weather_Clear
categorical__Weather_Foggy
categorical__Weather_Rainy
categorical__Weather_Snowy
categorical__Weather_Stormy
categorical__Road_Type_City Road
categorical__Road_Type_Highway
categorical__Road_Type_Mountain Road
categorical__Road_Type_Rural Road
categorical__Time_of_Day_Afternoon
categorical__Time_of_Day_Evening
categorical__Time_of_Day_Morning
categorical__Time_of_Day_Night
categorical__Road_Condition_Dry
categorical__Road_Condition_Icy
categorical__Road_Condition_Under Construction
categorical__Road_Condition_Wet
categorical__Vehicle_Type_Bus
categorical__Vehicle_Type_Car
categorical__Vehicle_Type_Motorcycle
categorical__Vehicle_Type_Truck
categorical__Road_Light_Condition_Artificial Light
categorical__Road_Light_Condition_Daylight
categorical__Road_Light_Condition_No Ligh

In [37]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)


In [38]:
print(
    "Missing values in processed training data:",
    X_train_processed_df.isnull().sum().sum()
)

print(
    "Missing values in processed testing data:",
    X_test_processed_df.isnull().sum().sum()
)

Missing values in processed training data: 0
Missing values in processed testing data: 0


In [40]:
print("Training target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())

Training target:
Accident
0    65
1    29
Name: count, dtype: int64

Testing target:
Accident
0    17
1     7
Name: count, dtype: int64


In [41]:
print("========== PREPROCESSING SUMMARY ==========")

print("Original dataset shape:", df.shape)

print("Training features:", X_train.shape)
print("Testing features :", X_test.shape)

print("Processed training features:", X_train_processed.shape)
print("Processed testing features :", X_test_processed.shape)

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTarget classes:")
print(y.value_counts())

print("\nPreprocessing completed successfully.")

========== PREPROCESSING SUMMARY ==========
Original dataset shape: (118, 14)
Training features: (94, 12)
Testing features : (24, 12)
Processed training features: (94, 30)
Processed testing features : (24, 30)

Numerical features:
['Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles', 'Driver_Alcohol', 'Driver_Age', 'Driver_Experience']

Categorical features:
['Weather', 'Road_Type', 'Time_of_Day', 'Road_Condition', 'Vehicle_Type', 'Road_Light_Condition']

Target classes:
Accident
0    82
1    36
Name: count, dtype: int64

Preprocessing completed successfully.
